In [ ]:
#Import các thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#Đọc file excel
df = pd.read_excel('/content/KQ_khaosat_da_xuly.xlsx')

# **BƯỚC 1: TIỀN XỬ LÝ DỮ LIỆU & TÍNH ĐIỂM THÀNH PHẦN (Data Preprocessing & Component Scoring)**
Mục tiêu của bước này là biến đổi các câu trả lời định tính từ bảng khảo sát thành dữ liệu định lượng (điểm số).

**1. Chuẩn hoá dữ liệu đầu vào:**

Làm sạch các dữ liệu trống (Missing values). Đối với các biến quan trọng như Thu nhập, Tài sản, nếu thiếu dữ liệu có thể điền bằng giá trị trung bình (Mean/Median) hoặc gán bằng 0 tuỳ nghiệp vụ.

Chuyển đổi câu trả lời trắc nghiệm thành điểm số (từ 1 đến 5) dựa trên bảng [link Bảng thang điểm chi tiết](https://docs.google.com/spreadsheets/d/126w9saVcENpa2at6r6JwHxzPjiOvMW6aTtWXBRJujCM/edit?gid=261714687#gid=261714687).

**2. Tính toán 6 Chỉ số lõi (Core Indices):**
Áp dụng trọng số để tính ra điểm tổng cho 6 nhóm chỉ số. Công thức tổng quát: Điểm Nhóm = Σ (Điểm Biến thành phần × Trọng số của biến đó).

- FHS (Sức khoẻ tài chính): = Thu nhập(0.1) + Chi tiêu(0.1) + Tích luỹ(0.15) + LIA(0.25) + DTA(0.15) + DPS(0.1) + Dòng tiền(0.15).

- ICS (Năng lực đầu tư): = KNW(0.5) + TIM(0.2) + PLN(0.3).

- RAS (Khẩu vị rủi ro): = ALLOC(0.35) + SL(0.25) + DROP(0.25) + MOT(0.15).

- PDS (Kỷ luật): = PLN(0.35) + TIM(0.2) + EXP(0.2) + SAV(0.15) + DPS(0.1).

- ANS (Nhu cầu hỗ trợ): = PAIN(0.4) + ADV(0.35) + PRI(0.25).

- CRS (Sẵn sàng chuyển đổi): = INIT(0.4) + CON(0.35) + FEE(0.25).

# **BƯỚC 2: KỸ THUẬT ĐẶC TRƯNG (Feature Engineering) - XÂY DỰNG MA TRẬN 2 TRỤC**

**1. Trục Y (Năng lực Tài chính & Đầu tư - Capability):**
Đo lường quy mô tài sản và trình độ của khách hàng.

`Y = (FHS * 0.6) + (ICS * 0.4)`

**Lý do:** Trọng số 0.6 nghiêng về FHS để đảm bảo hệ thống ưu tiên khách có tiền thực tế, kiến thức (ICS) đóng vai trò bổ trợ.

**2. Trục X (Độ nóng & Sẵn sàng - Readiness):**
Đo lường mức độ cấp thiết và khả năng chốt sale.

`X = (CRS * 0.6) + (ANS * 0.4)`

**Lý do:** CRS (Tiền và phí sẵn sàng trả) chiếm 60% để lọc những khách "chỉ kêu ca nỗi đau" (ANS) nhưng không chịu chi tiền.

# **BƯỚC 3: THIẾT LẬP LUẬT PHÂN CỤM (Rule-based Segmentation)**
Dùng ngưỡng điểm phân chia (Threshold) để đưa khách hàng vào 4 góc của ma trận. Ngưỡng tối ưu thường được chọn là 3.5 (mức độ từ khá đến xuất sắc trên thang 5).

Tập 1 - Ưu tiên (Y ≥ 3.5 & X ≥ 3.5): Khách có năng lực cao, sẵn sàng cao.

Tập 2 - Chiến lược (Y ≥ 3.5 & X < 3.5): Khách có năng lực cao, nhưng chưa sẵn sàng chuyển đổi.

Tập 3 - Tiêu chuẩn năng động (Y < 3.5 & X ≥ 3.5): Khách năng lực tài chính trung bình/thấp, nhưng rất khát khao đầu tư và cần hỗ trợ.

Tập 4 - Phễu Nuôi dưỡng / Bỏ Qua (Y < 3.5 & X < 3.5): Tách làm 2 luồng phụ:

4A (Nuôi dưỡng): Y < 3.5 & X < 3.5 nhưng ICS (Kiến thức) ≥ 3.0 HOẶC Tuổi ≤ 35.

4B (Bỏ qua/Nhiễu): Các trường hợp còn lại.

# **BƯỚC 4: KIỂM CHỨNG BẰNG THUẬT TOÁN HỌC MÁY (K-Means Clustering)**
Để đảm bảo luật phân cụm ở Bước 3 là chính xác và khách quan với dữ liệu thực tế, cần chạy thêm mô hình phân cụm không giám sát (Unsupervised Learning).

**Chuẩn hoá dữ liệu (Standardization):** Đưa 2 biến X, Y qua bộ lọc (ví dụ: StandardScaler trong Python) để đưa về cùng một tỷ lệ, tránh việc một biến có độ lệch chuẩn quá lớn lấn át biến kia.

**Khởi tạo thuật toán K-Means:** Thiết lập tham số K = 4 (tìm 4 cụm khách hàng).

**Huấn luyện & Trực quan hoá:** Máy học sẽ tự động gom các điểm dữ liệu gần nhau thành 4 tâm cụm (Centroids).


In [ ]:
def normalize_answer(answer):
    if pd.isna(answer):
        return ""
    return str(answer).strip().lower()

def get_score(value, mapping_dict, default_score=3):
    normalized_value = normalize_answer(value)
    for key, score in mapping_dict.items():
        if key.lower() == normalized_value:
            return score
    return default_score

def score_multi_choice_avg(value_str, mapping_dict, delimiter=';', default_score=3):
    if pd.isna(value_str) or value_str == "":
        return default_score
    choices = [normalize_answer(c) for c in value_str.split(delimiter) if c.strip()]
    if not choices: return default_score
    scores = []
    for choice in choices:
        found = False
        for key, score in mapping_dict.items():
            if key.lower() == choice:
                scores.append(score); found = True; break
        if not found: scores.append(default_score)
    return np.mean(scores) if scores else default_score

def score_fhs(row):
    # FHS = Thu nhập(0.1) + Chi tiêu(0.1) + Tích luỹ(0.15) + Tài sản nhàn rỗi(0.25) + Nợ/TS(0.15) + Thanh toán nợ(0.1) + Dòng tiền(0.15)
    s1 = get_score(row.get('Thu nhập'), {'Dưới 20 triệu VNĐ': 1, 'Từ 20 - 50 triệu VNĐ': 2, 'Từ 50 - 100 triệu VNĐ': 4, 'Từ 100 triệu VNĐ trở lên': 5, 'Không cố định': 3})
    s2 = get_score(row.get('Tỷ lệ chi tiêu'), {'Trên 80%': 1, '60–80%': 2, '40–60%': 3, 'Dưới 40%': 5, 'Chưa theo dõi cụ thể': 3})
    s3 = get_score(row.get('Tích luỹ'), {'Chưa có khoản dư đều đặn': 1, 'Dưới 10 triệu VNĐ': 2, 'Từ 10 - 30 triệu VNĐ': 3, 'Từ 30 - 50 triệu VNĐ': 4, 'Từ 50 triệu VNĐ trở lên': 5})
    s4 = get_score(row.get('Tài sản nhàn rỗi'), {'Chưa có khoản tiền nhàn rỗi': 1, 'Dưới 100 triệu VNĐ': 2, 'Từ 100 - 500 triệu VNĐ': 3, 'Từ 500 triệu - 1 tỷ VNĐ': 4, 'Từ 1 - 5 tỷ VNĐ': 5, 'Từ 5 tỷ VNĐ trở lên': 5})
    s5 = get_score(row.get('Tổng nợ/TS'), {'Trên 70%': 1, '50-70%': 2, '20-50%': 3, 'Dưới 20%': 5, 'Không tiện chia sẻ': 3})
    s6 = get_score(row.get('Tỷ lệ thanh toán Nợ'), {'Dưới 20%': 1, '20–40%': 2, '40–60%': 3, 'Trên 60%': 5, 'Chưa theo dõi cụ thể': 3})
    s7 = 3.0 # Dòng tiền - mặc định trung bình nếu chưa có biến cụ thể
    return (s1*0.1 + s2*0.1 + s3*0.15 + s4*0.25 + s5*0.15 + s6*0.1 + s7*0.15)

def score_ics(row):
    # ICS = KNW(0.5) + TIM(0.2) + PLN(0.3)
    knowledge_norm = normalize_answer(row.get('Hiểu biết đầu tư'))
    knw = 3
    if 'làm việc trong lĩnh vực tài chính' in knowledge_norm: knw = 5
    elif 'chứng chỉ' in knowledge_norm or 'tự tin' in knowledge_norm: knw = 4
    elif 'đã/đang đầu tư' in knowledge_norm: knw = 3
    elif 'cơ bản' in knowledge_norm: knw = 2
    elif 'không biết' in knowledge_norm: knw = 1

    tim = get_score(row.get('Thời điểm đầu tư'), {'Đầu tư khi có số tiền nhàn rỗi': 3, 'Đầu tư định kỳ hằng tháng/quý': 4, 'Kết hợp cả hai hình thức trên': 5})
    pln = get_score(row.get('Kế hoạch'), {'Chưa có kế hoạch rõ ràng': 1, 'Đã có kế hoạch nhưng chưa theo dõi': 3, 'Đã có kế hoạch và theo dõi': 5})
    return (knw*0.5 + tim*0.2 + pln*0.3)

def score_ras(row):
    # RAS = ALLOC(0.35) + SL(0.25) + DROP(0.25) + MOT(0.15)
    alloc = get_score(row.get('Phân bổ 1 tỷ giả định'), {'100% Tiền gửi tiết kiệm': 1, '60% Tiền gửi tiết kiệm, 30% Trái phiếu, 10% Cổ phiếu': 2, '100% Trái phiếu': 2, 'Phân bổ đều vào cả 3 kênh': 3, '10% Tiền gửi tiết kiệm, 30% Trái phiếu, 60% Cổ phiếu': 4, '100% Cổ phiếu': 5})
    sl = get_score(row.get('Ngưỡng cắt lỗ'), {'Dưới 5%': 1, 'Từ 5% đến dưới 15%': 2, 'Không tiện chia sẻ': 3, 'Từ 15% đến dưới 30%': 4, 'Từ 30% trở lên': 5})
    drop = get_score(row.get('Giảm 15%'), {'Muốn bán bớt ngay': 1, 'Hoang mang': 1, 'Tạm dừng theo dõi': 2, 'Tôi chưa từng ở tình huống này': 3, 'Cân nhắc mua thêm': 5})
    mot = get_score(row.get('Lý do chọn kênh đầu tư'), {'Ưu tiên bảo toàn vốn': 1, 'Tạo dòng tiền ổn định': 2, 'Chưa rõ mục tiêu': 3, 'Tăng trưởng tài sản đều đặn': 4, 'Chấp nhận biến động cao': 5})
    return (alloc*0.35 + sl*0.25 + drop*0.25 + mot*0.15)

def score_pds(row):
    # PDS = PLN(0.35) + TIM(0.2) + EXP(0.2) + SAV(0.15) + DPS(0.1)
    pln = get_score(row.get('Kế hoạch'), {'Chưa có kế hoạch rõ ràng': 1, 'Đã có kế hoạch nhưng chưa theo dõi': 3, 'Đã có kế hoạch và theo dõi': 5})
    tim = get_score(row.get('Thời điểm đầu tư'), {'Đầu tư khi có số tiền nhàn rỗi': 3, 'Đầu tư định kỳ hằng tháng/quý': 4, 'Kết hợp cả hai hình thức trên': 5})
    exp = get_score(row.get('Tỷ lệ chi tiêu'), {'Trên 80%': 1, '60–80%': 2, '40–60%': 3, 'Dưới 40%': 5, 'Chưa theo dõi cụ thể': 3})
    sav = get_score(row.get('Tích luỹ'), {'Chưa có khoản dư đều đặn': 1, 'Dưới 10 triệu VNĐ': 2, 'Từ 10 - 30 triệu VNĐ': 3, 'Từ 30 - 50 triệu VNĐ': 4, 'Từ 50 triệu VNĐ trở lên': 5})
    dps = get_score(row.get('Tỷ lệ thanh toán Nợ'), {'Dưới 20%': 1, '20–40%': 2, '40–60%': 3, 'Trên 60%': 5, 'Chưa theo dõi cụ thể': 3})
    return (pln*0.35 + tim*0.2 + exp*0.2 + sav*0.15 + dps*0.1)

def score_ans(row):
    # ANS = PAIN(0.4) + ADV(0.35) + PRI(0.25)
    pain = score_multi_choice_avg(row.get('Khó khăn'), {'Việc đầu tư hiện tại đạt kỳ vọng': 1, 'Chưa có kế hoạch tài chính/tài sản rõ ràng': 4, 'Chưa biết phân bổ tài sản': 5, 'Quá nhiều thông tin': 4, 'Không có đủ thời gian': 4, 'Thiếu tự tin': 5, 'Chưa tìm được người tư vấn đáng tin cậy': 5})
    adv = get_score(row.get('Lựa chọn yên tâm'), {'Chủ động thực hiện': 1, 'Chủ động thực hiện nhưng có tham khảo': 3, 'Ủy thác đầu tư hoàn toàn vào hệ thống': 4, 'Ủy thác đầu từ hoàn toàn cho chuyên gia': 5})
    pri = 3.0 # PRI (Ưu tiên hỗ trợ) - mặc định
    return (pain*0.4 + adv*0.35 + pri*0.25)

def score_crs(row):
    # CRS = INIT(0.4) + CON(0.35) + FEE(0.25)
    init = get_score(row.get('Số tiền ban đầu'), {'Dưới 10 triệu VNĐ': 2, 'Từ 10 - 50 triệu VNĐ': 3, 'Từ 50 - 200 triệu VNĐ': 4, 'Từ 200 triệu VNĐ trở lên': 5})
    con = get_score(row.get('Sẵn sàng trao đổi'), {'Không': 1, 'Có thể': 3, 'Có': 5})
    fee = get_score(row.get('Phí sẵn sàng trả'), {'0.5% / năm trở xuống': 2, 'Trên 0.5% đến 1%': 3, 'Trên 1% đến 2%': 4, 'Trên 2%': 5})
    return (init*0.4 + con*0.35 + fee*0.25)

# Main execution
df['FHS'] = df.apply(score_fhs, axis=1)
df['ICS'] = df.apply(score_ics, axis=1)
df['RAS'] = df.apply(score_ras, axis=1)
df['PDS'] = df.apply(score_pds, axis=1)
df['ANS'] = df.apply(score_ans, axis=1)
df['CRS'] = df.apply(score_crs, axis=1)

# Feature Engineering
df['Y_Capability'] = (df['FHS'] * 0.6) + (df['ICS'] * 0.4)
df['X_Readiness'] = (df['CRS'] * 0.6) + (df['ANS'] * 0.4)

def rule_based_segmentation(row):
    y, x = row['Y_Capability'], row['X_Readiness']
    ics = row['ICS']
    age_val = row.get('Độ tuổi', '')
    is_young = False
    if any(k in str(age_val) for k in ['Dưới 18', '18 đến 25', '26 đến 35']):
        is_young = True

    if y >= 3.5 and x >= 3.5:
        return "Khách hàng Ưu tiên (Priority / Premier Clients)"
    elif y >= 3.5 and x < 3.5:
        return "Khách hàng Chiến lược (Strategic Accounts)"
    elif y < 3.5 and x >= 3.5:
        return "Khách hàng Tiêu chuẩn Năng động (Active Standard Clients)"
    else:
        if ics >= 3.0 or is_young:
            return "Khách hàng Ươm tạo (Incubation Leads)"
        return "Khách hàng Ngoài mục tiêu (Out-of-target / Archived)"

df['Segment'] = df.apply(rule_based_segmentation, axis=1)

print(df['Segment'].value_counts())
display(df[['Y_Capability', 'X_Readiness', 'Segment']].head())

Segment
Khách hàng Ươm tạo (Incubation Leads)                        294
Khách hàng Chiến lược (Strategic Accounts)                    67
Khách hàng Ngoài mục tiêu (Out-of-target / Archived)          20
Khách hàng Tiêu chuẩn Năng động (Active Standard Clients)      7
Khách hàng Ưu tiên (Priority / Premier Clients)                4
Name: count, dtype: int64


,Y_Capability,X_Readiness,Segment
0,3.46,3.143333,Khách hàng Ươm tạo (Incubation Leads)
1,3.13,3.240000,Khách hàng Ươm tạo (Incubation Leads)
2,3.53,2.610000,Khách hàng Chiến lược (Strategic Accounts)
3,4.05,3.090000,Khách hàng Chiến lược (Strategic Accounts)
4,3.08,3.240000,Khách hàng Ươm tạo (Incubation Leads)


In [ ]:
segment_counts = df["Segment"].value_counts(dropna=False)

print(segment_counts)

print("Tổng khách hàng:", len(df))

print(
    "Tổng đã phân khúc:",
    df["Segment"].notna().sum()
)

print(
    "Số REVIEW:",
    (df["Segment"] == "REVIEW").sum()
)

print(
    "Số giá trị thiếu:",
    df["Segment"].isna().sum()
)

# The following assert statements are removed as they were causing errors due to the presence of 'REVIEW' segment.
# assert len(df) == df["Phân khúc khách hàng"].notna().sum()

# assert (
#     (df["Phân khúc khách hàng"] == "REVIEW").sum()
#     == 0
# )

# assert df["Phân khúc khách hàng"].isna().sum() == 0

Segment
Khách hàng Ươm tạo (Incubation Leads)                        294
Khách hàng Chiến lược (Strategic Accounts)                    67
Khách hàng Ngoài mục tiêu (Out-of-target / Archived)          20
Khách hàng Tiêu chuẩn Năng động (Active Standard Clients)      7
Khách hàng Ưu tiên (Priority / Premier Clients)                4
Name: count, dtype: int64
Tổng khách hàng: 392
Tổng đã phân khúc: 392
Số REVIEW: 0
Số giá trị thiếu: 0


In [ ]:
summary = (
    df["Segment"]
    .value_counts()
    .rename_axis("Phân khúc")
    .reset_index(name="Số lượng")
)

summary["Tỷ lệ (%)"] = (
    summary["Số lượng"] / len(df) * 100
).round(2)

print(summary)

                                           Phân khúc  Số lượng  Tỷ lệ (%)
0              Khách hàng Ươm tạo (Incubation Leads)       294      75.00
1         Khách hàng Chiến lược (Strategic Accounts)        67      17.09
2  Khách hàng Ngoài mục tiêu (Out-of-target / Arc...        20       5.10
3  Khách hàng Tiêu chuẩn Năng động (Active Standa...         7       1.79
4    Khách hàng Ưu tiên (Priority / Premier Clients)         4       1.02


In [ ]:
profile = (
    df.groupby("Segment")[
        ["FHS", "ICS", "RAS", "PDS", "ANS", "CRS"]
    ]
    .mean()
    .round(2)
)

print(profile)

                                                     FHS   ICS   RAS   PDS  \
Segment                                                                      
Khách hàng Chiến lược (Strategic Accounts)          3.39  4.22  2.89  3.39   
Khách hàng Ngoài mục tiêu (Out-of-target / Arch...  2.76  2.52  2.92  3.02   
Khách hàng Tiêu chuẩn Năng động (Active Standar...  3.06  3.33  2.96  3.11   
Khách hàng Ưu tiên (Priority / Premier Clients)     3.42  4.40  3.00  3.50   
Khách hàng Ươm tạo (Incubation Leads)               2.64  3.74  2.94  2.95   

                                                     ANS   CRS  
Segment                                                         
Khách hàng Chiến lược (Strategic Accounts)          3.04  3.02  
Khách hàng Ngoài mục tiêu (Out-of-target / Arch...  3.07  2.68  
Khách hàng Tiêu chuẩn Năng động (Active Standar...  3.61  3.71  
Khách hàng Ưu tiên (Priority / Premier Clients)     3.69  3.80  
Khách hàng Ươm tạo (Incubation Leads)               3.08  2.82 

In [ ]:
score_stats = df.groupby('Segment')[['FHS', 'ICS', 'RAS', 'PDS', 'ANS', 'CRS']].agg(['min', 'max']).round(2)

print("Thống kê Min/Max theo từng nhóm điểm và phân khúc:")
display(score_stats)

Thống kê Min/Max theo từng nhóm điểm và phân khúc:


FHS        ICS       \
                                                     min   max  min  max   
Segment                                                                    
Khách hàng Chiến lược (Strategic Accounts)          2.90  4.30  3.4  4.4   
Khách hàng Ngoài mục tiêu (Out-of-target / Arch...  2.35  3.95  2.0  2.9   
Khách hàng Tiêu chuẩn Năng động (Active Standar...  2.30  3.70  2.0  4.4   
Khách hàng Ưu tiên (Priority / Premier Clients)     2.90  3.70  4.4  4.4   
Khách hàng Ươm tạo (Incubation Leads)               1.95  4.10  2.0  4.4   

                                                     RAS         PDS        \
                                                     min   max   min   max   
Segment                                                                      
Khách hàng Chiến lược (Strategic Accounts)          2.50  3.50  2.80  4.10   
Khách hàng Ngoài mục tiêu (Out-of-target / Arch...  2.50  3.50  2.25  3.95   
Khách hàng Tiêu chuẩn Năng động (Active Standar...  2.50  3.50  2.45  3.40   
Khách hàng Ưu tiên (Priority / Premier Clients)     2.75  3.25  3.30  3.80   
Khách hàng Ươm tạo (Incubation Leads)               2.50  3.50  2.25  4.10   

                                                     ANS         CRS        
                                                     min   max   min   max  
Segment                                                                     
Khách hàng Chiến lược (Strategic Accounts)          3.00  3.70  2.35  3.80  
Khách hàng Ngoài mục tiêu (Out-of-target / Arch...  3.00  3.40  2.35  3.55  
Khách hàng Tiêu chuẩn Năng động (Active Standar...  3.20  4.10  3.40  3.80  
Khách hàng Ưu tiên (Priority / Premier Clients)     3.13  4.10  3.80  3.80  
Khách hàng Ươm tạo (Incubation Leads)               3.00  3.83  2.35  3.80

In [ ]:
# Xuất file kết quả phân khúc khách hàng chuyên nghiệp
output_filename = 'ket_qua_phan_khuc_chuyen_nghiep.xlsx'
df.to_excel(output_filename, index=False)
print(f"Đã xuất file: {output_filename}")

Đã xuất file: ket_qua_phan_khuc_chuyen_nghiep.xlsx
